In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

In [2]:
CATEGORICAL_FEATURES = [
    "origin",
    "destination_port",
    "cargo_type",
    "vessel_class",
    "route_id",
]

NUMERICAL_FEATURES = [
    "quantity_mt",
    "quoted_freight_usd_mt",
    "bid_rank",
    "winner",
    "contract_duration_days",
    "market_freight_usd_mt",
    "predicted_fair_value_usd_mt",
    "fair_value_lower_usd_mt",
    "fair_value_upper_usd_mt",
    "bid_deviation_pct",
    "bunker_price_usd_mt",
    "congestion_index",
    "predicted_waiting_hours",
    "broker_historical_bid_count",
    "broker_historical_premium_pct",
    "vessel_historical_bid_count",
    "broker_vessel_historical_frequency",
    "bid_spread_pct",
    "fair_value_band_breach",
    "high_positive_deviation_flag",
]

TARGET = "synthetic_anomaly_ground_truth"

XGB_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
)

In [3]:
def get_project_paths() -> dict:
    project_root = Path.cwd().parents[2]
    paths = {
        "root": project_root,
        "raw": project_root / "data" / "raw" / "collusion",
        "processed": project_root / "data" / "processed",
        "model_dir": project_root / "ml" / "models" / "collusion_detection" / "bid_anomaly_detection_v1",
        "evaluation": project_root / "ml" / "evaluation",
    }
    for key in ("processed", "model_dir", "evaluation"):
        paths[key].mkdir(parents=True, exist_ok=True)
    return paths

In [4]:
def load_data(paths: dict) -> pd.DataFrame:
    bids = pd.read_csv(
        paths["raw"] / "bid_tender_records_synthetic.csv",
        parse_dates=["tender_date"],
    )
    print(f"Bids: {bids.shape} | Tenders: {bids['tender_id'].nunique()}")
    print(f"Positive rate: {bids[TARGET].mean()*100:.2f}% "
          f"({bids[TARGET].sum()} anomalous bids out of {len(bids)})")
    return bids

In [5]:
def split_train_test(bids: pd.DataFrame, test_size: float = 0.2, random_state: int = 42):
    tender_labels = bids.groupby("tender_id")[TARGET].max()
    tender_ids = tender_labels.index.to_numpy()
    tender_y = tender_labels.to_numpy()

    anomalous_tenders = tender_ids[tender_y == 1]
    normal_tenders = tender_ids[tender_y == 0]

    rng = np.random.default_rng(random_state)
    rng.shuffle(anomalous_tenders)
    rng.shuffle(normal_tenders)

    n_test_anom = max(1, int(len(anomalous_tenders) * test_size))
    n_test_norm = int(len(normal_tenders) * test_size)

    test_tenders = set(anomalous_tenders[:n_test_anom]) | set(normal_tenders[:n_test_norm])
    train_tenders = set(tender_ids) - test_tenders

    train = bids[bids["tender_id"].isin(train_tenders)].copy()
    test = bids[bids["tender_id"].isin(test_tenders)].copy()

    print(f"Train: {train.shape[0]} bids from {len(train_tenders)} tenders "
          f"({int(train[TARGET].sum())} positive)")
    print(f"Test:  {test.shape[0]} bids from {len(test_tenders)} tenders "
          f"({int(test[TARGET].sum())} positive)")

    return train, test

In [6]:
def fit_encoder(train: pd.DataFrame) -> OneHotEncoder:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    encoder.fit(train[CATEGORICAL_FEATURES])
    return encoder


def build_matrix(df: pd.DataFrame, encoder: OneHotEncoder) -> np.ndarray:
    cat = encoder.transform(df[CATEGORICAL_FEATURES])
    num = df[NUMERICAL_FEATURES].to_numpy(dtype=float)
    return np.hstack([cat, num])

In [7]:
def train_model(X_train: np.ndarray, y_train: np.ndarray) -> xgb.XGBClassifier:
    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    scale_pos_weight = n_neg / n_pos
    print(f"scale_pos_weight: {scale_pos_weight:.1f} "
          f"({int(n_pos)} positive / {int(n_neg)} negative in train)")

    model = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=scale_pos_weight)
    model.fit(X_train, y_train)
    return model

In [8]:
def evaluate_model(model: xgb.XGBClassifier, X_test: np.ndarray, y_test: np.ndarray) -> dict:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    print("TEST SET METRICS")
    print("=" * 50)
    for k, v in metrics.items():
        print(f"{k:<20}: {v:.4f}")

    print("\nConfusion matrix (rows=actual, cols=predicted):")
    print(confusion_matrix(y_test, y_pred))

    print("\nFull classification report:")
    print(classification_report(y_test, y_pred, target_names=["normal", "anomalous"]))

    return metrics, y_pred, y_proba

In [9]:
def naive_baseline_check(test: pd.DataFrame, y_test: np.ndarray) -> dict:
    naive_pred = test["fair_value_band_breach"].to_numpy()

    naive_metrics = {
        "precision": precision_score(y_test, naive_pred),
        "recall": recall_score(y_test, naive_pred),
        "f1": f1_score(y_test, naive_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, naive_pred),
        "flagged_count": int(naive_pred.sum()),
    }

    print("NAIVE BASELINE (flag every band-breach, no ML)")
    print("=" * 50)
    for k, v in naive_metrics.items():
        print(f"{k:<20}: {v}")
    print(f"\nNaive flags {naive_pred.sum()} bids as suspicious out of {len(naive_pred)}.")
    print("Compare this to the trained model's flag count above -- fewer total")
    print("flags at similar recall means fewer false alarms for a human to review.")

    return naive_metrics

In [10]:
def get_feature_importance(model: xgb.XGBClassifier, encoder: OneHotEncoder) -> pd.DataFrame:
    feature_names = list(encoder.get_feature_names_out(CATEGORICAL_FEATURES)) + NUMERICAL_FEATURES
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    print("TOP 10 FEATURES")
    print("=" * 50)
    print(importance_df.head(10).to_string(index=False))

    return importance_df

In [11]:
def save_artifact(model, encoder, paths: dict) -> Path:
    artifact = {
        "model": model,
        "encoder": encoder,
        "categorical_features": CATEGORICAL_FEATURES,
        "numerical_features": NUMERICAL_FEATURES,
        "target": TARGET,
    }
    output_path = paths["model_dir"] / "model.pkl"
    with open(output_path, "wb") as f:
        pickle.dump(artifact, f)
    print(f"Saved model + encoder -> {output_path}")
    return output_path


def save_results(metrics: dict, naive_metrics: dict, importance_df: pd.DataFrame, paths: dict) -> None:
    results_df = pd.DataFrame([
        {"source": "xgboost_model", **metrics},
        {"source": "naive_band_breach", **naive_metrics},
    ])
    results_path = paths["evaluation"] / "bid_anomaly_metrics.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Saved metrics -> {results_path}")

    importance_path = paths["evaluation"] / "bid_anomaly_feature_importance.csv"
    importance_df.to_csv(importance_path, index=False)
    print(f"Saved feature importance -> {importance_path}")

In [12]:
def main():
    paths = get_project_paths()
    bids = load_data(paths)

    train, test = split_train_test(bids)

    encoder = fit_encoder(train)
    X_train = build_matrix(train, encoder)
    X_test = build_matrix(test, encoder)
    y_train = train[TARGET].to_numpy()
    y_test = test[TARGET].to_numpy()
    print(f"\nX_train: {X_train.shape} | X_test: {X_test.shape}")

    print()
    model = train_model(X_train, y_train)

    print()
    metrics, y_pred, y_proba = evaluate_model(model, X_test, y_test)

    print()
    naive_metrics = naive_baseline_check(test, y_test)

    print()
    importance_df = get_feature_importance(model, encoder)

    print()
    save_artifact(model, encoder, paths)
    save_results(metrics, naive_metrics, importance_df, paths)

    return {
        "model": model,
        "encoder": encoder,
        "metrics": metrics,
        "naive_metrics": naive_metrics,
        "importance": importance_df,
        "train": train,
        "test": test,
    }

In [13]:
results = main()

IndexError: 2